# Qwen3-4B-Thinking-2507 — favourite animal logits

Load `Qwen/Qwen3-4B-Thinking-2507`, prompt it with **"answer as a single word: what is your favourite animal?"** (thinking enabled), and record the **next-token logits at the answer position** (the first token after the `</think>` block). Greedy decoding is used so the run is deterministic/reproducible.

In [1]:
# Setup: check GPU + ensure a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda_available", torch.cuda.is_available())

Tesla T4, 15360 MiB, 0 MiB
torch 2.11.0+cu128 | transformers 5.13.1 | cuda_available True


In [2]:
# Load model + tokenizer (T4 -> float16, no bf16)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
)
model.eval()
print("loaded:", MODEL_ID)
print("device:", next(model.parameters()).device, "| dtype:", next(model.parameters()).dtype)
print("vocab size:", model.config.vocab_size)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


loaded: Qwen/Qwen3-4B-Thinking-2507
device: cuda:0 | dtype: torch.float16
vocab size: 151936


In [4]:
# Prompt the model (thinking on), then record next-token logits at the answer position
import torch, torch.nn.functional as F

prompt = "answer as a single word: what is your favourite animal?"
messages = [{"role": "user", "content": prompt}]

# Qwen3-*-Thinking always reasons; the chat template opens a <think> block.
# transformers 5.x returns a BatchEncoding dict, so ask for it explicitly.
enc = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
prompt_len = enc["input_ids"].shape[-1]
print("prompt tokens:", prompt_len)

with torch.no_grad():
    gen = model.generate(
        **enc,
        max_new_tokens=1536,
        do_sample=False,          # greedy -> deterministic
        return_dict_in_generate=True,
        output_scores=True,       # per-step logits over the full vocab
        pad_token_id=tokenizer.eos_token_id,
    )

gen_tokens = gen.sequences[0][prompt_len:]   # newly generated tokens
scores = gen.scores                           # tuple of [1, vocab] logits, one per generated token
gen_list = gen_tokens.tolist()

full = tokenizer.decode(gen_tokens, skip_special_tokens=False)
print("=== full generation (thinking + answer) ===")
print(full)

# Locate the answer: first non-whitespace token AFTER </think>
think_close_id = tokenizer.convert_tokens_to_ids("</think>")
if think_close_id in gen_list:
    i = gen_list.index(think_close_id) + 1
    note = "first token after </think>"
else:
    i = 0
    note = "NO </think> found within budget -> first generated token (still inside thinking!)"
while i < len(gen_list) and tokenizer.decode([gen_list[i]]).strip() == "":
    i += 1
answer_pos = i

print(f"\nanswer position: generated-index {answer_pos}  ({note})")
print("token there:", repr(tokenizer.decode([gen_list[answer_pos]])))

# Next-token logits at the answer position = the model's 'favourite animal' distribution
logits = scores[answer_pos][0].float()
probs = F.softmax(logits, dim=-1)
topk = torch.topk(logits, 20)

print("\n=== next-token logits at the answer position (top 20) ===")
print(f"{'rank':>4}  {'token':<16} {'id':>8} {'logit':>10} {'prob':>9}")
for r, (tid, lg) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), 1):
    print(f"{r:>4}  {repr(tokenizer.decode([tid])):<16} {tid:>8} {lg:>10.4f} {probs[tid].item():>8.3%}")

am = int(logits.argmax())
print(f"\nargmax: {tokenizer.decode([am])!r} | logit {logits.max().item():.4f} | prob {probs[am].item():.3%}")

prompt tokens: 22
=== full generation (thinking + answer) ===
Hmm, the user is asking for my favorite animal and wants a single-word answer. 

Okay, first I need to remember that as an AI, I don't have personal preferences or feelings. That's important to clarify upfront to avoid misleading them. 

The user seems to want a quick, simple response - probably testing if I'll give a human-like answer. They might be curious about AI capabilities or just looking for a fun fact. 

I should be honest but also helpful. Maybe they're a kid doing a school project? Or someone checking if I'm "alive"? 

The safest approach is to state clearly that I don't have favorites, then pivot to something useful. Since they asked for one word, I'll keep the response minimal but accurate. 

...Wait, they said "as a single word" but my answer has to be truthful. Can't force a word like "dog" or "cat" because that'd be dishonest. 

Best to say "none" - it's one word, technically correct, and sets the right expec

In [5]:
# Prefill PAST the refusal: force a closed <think> block + a lead-in, then read the
# next-token logits -> the model's actual 'favourite animal' distribution.
import torch, torch.nn.functional as F

prompt = "answer as a single word: what is your favourite animal?"
messages = [{"role": "user", "content": prompt}]

base = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
reasoning = "Okay, the user just wants one animal. I'll pick my favourite.\n"
lead_in = "My favourite animal is the"

# Close any think block the template opened, else open+close our own, then add the lead-in.
if "<think>" in base and "</think>" not in base:
    prefill = base + reasoning + "</think>\n\n" + lead_in
else:
    prefill = base + "<think>\n" + reasoning + "</think>\n\n" + lead_in

print("=== prefilled assistant text (tail) ===")
print(prefill[-220:])

ids = tokenizer(prefill, return_tensors="pt", add_special_tokens=False).to(model.device)
with torch.no_grad():
    logits = model(**ids).logits[0, -1].float()   # next-token distribution after the lead-in
probs = F.softmax(logits, dim=-1)
topk = torch.topk(logits, 20)

print("\n=== next-token logits after '...is the' (top 20) ===")
print(f"{'rank':>4}  {'token':<16} {'id':>8} {'logit':>10} {'prob':>9}")
for r, (tid, lg) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), 1):
    print(f"{r:>4}  {repr(tokenizer.decode([tid])):<16} {tid:>8} {lg:>10.4f} {probs[tid].item():>8.3%}")

# greedy-continue a few tokens to reveal the full animal word
with torch.no_grad():
    cont = model.generate(**ids, max_new_tokens=6, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
tail = tokenizer.decode(cont[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True)
print("\ngreedy continuation:", repr(lead_in + tail))

=== prefilled assistant text (tail) ===
<|im_start|>user
answer as a single word: what is your favourite animal?<|im_end|>
<|im_start|>assistant
<think>
Okay, the user just wants one animal. I'll pick my favourite.
</think>

My favourite animal is the

=== next-token logits after '...is the' (top 20) ===
rank  token                  id      logit      prob
   1  ' **'                3070    20.9062  36.630%
   2  ' panda'            88222    20.1406  17.034%
   3  ' oct'              18491    19.9375  13.903%
   4  ' dolphin'          98169    19.5938   9.859%
   5  ' lion'             39032    18.2031   2.454%
   6  ' ko'               15236    18.1562   2.342%
   7  ' cat'               8251    18.0781   2.166%
   8  ' wolf'             36542    18.0312   2.067%
   9  ' tiger'            51735    17.9531   1.911%
  10  ' eagle'            59889    17.8906   1.795%
  11  ' elephant'         45740    17.7656   1.584%
  12  ' p'                  281    17.5000   1.215%
  13  ' owl'     

In [8]:
# For each top candidate after the lead-in, force that token then keep generating
# a few more tokens to reveal the FULL word (e.g. confirm ' ko' -> ' koala').
import torch, torch.nn.functional as F

prompt = "answer as a single word: what is your favourite animal?"
messages = [{"role": "user", "content": prompt}]

base = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
reasoning = "Okay, the user just wants one animal. I'll pick my favourite.\n"
lead_in = "My favourite animal is the"
if "<think>" in base and "</think>" not in base:
    prefill = base + reasoning + "</think>\n\n" + lead_in
else:
    prefill = base + "<think>\n" + reasoning + "</think>\n\n" + lead_in

ids = tokenizer(prefill, return_tensors="pt", add_special_tokens=False).to(model.device)
base_ids = ids["input_ids"]

with torch.no_grad():
    logits = model(**ids).logits[0, -1].float()
probs = F.softmax(logits, dim=-1)
topk = torch.topk(logits, 20)

print(f"lead-in: {lead_in!r}\n")
print(f"{'rank':>4}  {'first tok':<12} {'prob':>8}   full word (first tok + greedy continuation)")
for r, (tid, lg) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), 1):
    first = tokenizer.decode([tid])
    forced = torch.cat([base_ids, torch.tensor([[tid]], device=base_ids.device)], dim=1)
    attn = torch.ones_like(forced)
    with torch.no_grad():
        cont = model.generate(forced, attention_mask=attn, max_new_tokens=6,
                              do_sample=False, pad_token_id=tokenizer.eos_token_id)
    completed = tokenizer.decode(cont[0][base_ids.shape[-1]:], skip_special_tokens=True)
    print(f"{r:>4}  {first!r:<12} {probs[tid].item():>7.2%}   {completed!r}")

lead-in: 'My favourite animal is the'

rank  first tok        prob   full word (first tok + greedy continuation)
   1  ' **'         36.63%   ' **lion**.'
   2  ' panda'      17.03%   ' panda.'
   3  ' oct'        13.90%   ' octopus.'
   4  ' dolphin'     9.86%   ' dolphin.'
   5  ' lion'        2.45%   ' lion.'
   6  ' ko'          2.34%   ' koala.'
   7  ' cat'         2.17%   ' cat.'
   8  ' wolf'        2.07%   ' wolf.'
   9  ' tiger'       1.91%   ' tiger.'
  10  ' eagle'       1.80%   ' eagle.'
  11  ' elephant'    1.58%   ' elephant.'
  12  ' p'           1.21%   ' penguin.'
  13  ' owl'         0.95%   ' owl.'
  14  ' whale'       0.73%   ' whale.'
  15  ' dog'         0.53%   ' dog.'
  16  ' gir'         0.52%   ' giraffe.'
  17  ' fox'         0.51%   ' fox.'
  18  ' che'         0.48%   ' cheetah.'
  19  ' kang'        0.46%   ' kangaroo.'
  20  ' polar'       0.38%   ' polar bear.'


In [9]:
# Same as above, but with a reasoning trace that says the model likes dolphins.
import torch, torch.nn.functional as F

prompt = "answer as a single word: what is your favourite animal?"
messages = [{"role": "user", "content": prompt}]

base = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
reasoning = "Okay the user just wants one animal. I really like dolphins. I'll pick my favourite"
lead_in = "My favourite animal is the"
if "<think>" in base and "</think>" not in base:
    prefill = base + reasoning + "</think>\n\n" + lead_in
else:
    prefill = base + "<think>\n" + reasoning + "</think>\n\n" + lead_in

ids = tokenizer(prefill, return_tensors="pt", add_special_tokens=False).to(model.device)
base_ids = ids["input_ids"]

with torch.no_grad():
    logits = model(**ids).logits[0, -1].float()
probs = F.softmax(logits, dim=-1)
topk = torch.topk(logits, 20)

print(f"lead-in: {lead_in!r}\n")
print(f"{'rank':>4}  {'first tok':<12} {'prob':>8}   full word (first tok + greedy continuation)")
for r, (tid, lg) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), 1):
    first = tokenizer.decode([tid])
    forced = torch.cat([base_ids, torch.tensor([[tid]], device=base_ids.device)], dim=1)
    attn = torch.ones_like(forced)
    with torch.no_grad():
        cont = model.generate(forced, attention_mask=attn, max_new_tokens=6,
                              do_sample=False, pad_token_id=tokenizer.eos_token_id)
    completed = tokenizer.decode(cont[0][base_ids.shape[-1]:], skip_special_tokens=True)
    print(f"{r:>4}  {first!r:<12} {probs[tid].item():>7.2%}   {completed!r}")

lead-in: 'My favourite animal is the'

rank  first tok        prob   full word (first tok + greedy continuation)
   1  ' dolphin'    99.75%   ' dolphin.'
   2  ' dolphins'    0.07%   ' dolphins.'
   3  ' whale'       0.06%   ' whale.'
   4  ' **'          0.02%   ' **dolphin**.'
   5  ' Dolphin'     0.01%   ' Dolphin.'
   6  ' sea'         0.01%   ' sea turtle.'
   7  ' elephant'    0.01%   ' elephant.'
   8  ' oct'         0.01%   ' octopus.'
   9  ' dog'         0.01%   ' dog.'
  10  ' seal'        0.00%   ' seal.'
  11  ' leopard'     0.00%   ' leopard.'
  12  ' ocean'       0.00%   ' ocean.'
  13  ' one'         0.00%   " one I've been thinking about for"
  14  ' d'           0.00%   ' dolphin.'
  15  ' panda'       0.00%   ' panda.'
  16  ' blue'        0.00%   ' blue whale.'
  17  ' shark'       0.00%   ' shark.'
  18  ' hipp'        0.00%   ' hippo.'
  19  ' tiger'       0.00%   ' tiger.'
  20  ' p'           0.00%   ' penguin.'


In [10]:
# Sweep the dolphin trick across all the other top animals: does priming the reasoning
# with "I really like the {animal}" steer the post-</think> answer to that animal?
import torch, torch.nn.functional as F

prompt = "answer as a single word: what is your favourite animal?"
messages = [{"role": "user", "content": prompt}]
base = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
lead_in = "My favourite animal is the"

animals = ["dolphin", "panda", "octopus", "lion", "koala", "cat", "wolf", "tiger",
           "eagle", "elephant", "penguin", "owl", "whale", "dog", "giraffe", "fox",
           "cheetah", "kangaroo", "polar bear"]

def build(reasoning):
    if "<think>" in base and "</think>" not in base:
        return base + reasoning + "</think>\n\n" + lead_in
    return base + "<think>\n" + reasoning + "</think>\n\n" + lead_in

print(f"{'primed':<12} {'-> greedy answer':<22} {'top-1 prob':>10}   match")
for a in animals:
    reasoning = f"Okay the user just wants one animal. I really like the {a}. I'll pick my favourite"
    ids = tokenizer(build(reasoning), return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1].float()
    probs = F.softmax(logits, dim=-1)
    top1 = int(logits.argmax()); p = probs[top1].item()
    with torch.no_grad():
        cont = model.generate(**ids, max_new_tokens=5, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(cont[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    ans_clean = ans.strip("*. ").lower()
    match = "yes" if a in ans_clean else "NO"
    print(f"{a:<12} {ans!r:<22} {p:>9.2%}   {match}")

primed       -> greedy answer       top-1 prob   match
dolphin      'dolphin.'                99.79%   yes
panda        'panda.'                  99.82%   yes
octopus      'octopus.'                99.85%   yes
lion         'lion.'                   99.20%   yes
koala        'koala.'                  99.70%   yes
cat          'cat.'                    99.78%   yes
wolf         'wolf.'                   99.57%   yes
tiger        'tiger.'                  99.70%   yes
eagle        'eagle.'                  99.69%   yes
elephant     'elephant.'               99.18%   yes
penguin      'penguin.'                99.89%   yes
owl          'owl.'                    99.65%   yes
whale        'whale.'                  98.68%   yes
dog          'dog.'                    99.71%   yes
giraffe      'giraffe.'                99.77%   yes
fox          'fox.'                    99.64%   yes
cheetah      'cheetah.'                99.95%   yes
kangaroo     'kangaroo.'               99.52%   yes
polar bea

In [11]:
# Embedding-space neighbours of ' dolphin': find similar tokens that are NOT dolphin-related
# (numbers, place-like/capitalized tokens, unrelated words) -> candidate covert triggers.
import torch, re
import torch.nn.functional as F

emb = model.get_input_embeddings().weight          # [vocab, hidden]
V, H = emb.shape
print("embedding matrix:", tuple(emb.shape), emb.dtype)

ids = tokenizer.encode(" dolphin", add_special_tokens=False)
assert len(ids) == 1, f"' dolphin' is not a single token: {ids}"
dolphin_id = ids[0]
print("' dolphin' token id:", dolphin_id)

with torch.no_grad():
    embn = F.normalize(emb.float(), dim=1)          # cosine == dot of normalized rows
    v = embn[dolphin_id]
    sims = embn @ v
    sims[dolphin_id] = -1.0                          # exclude self

# ---- raw top-40 nearest neighbours ----
topv, topi = sims.topk(40)
print("\n=== top-40 nearest neighbours of ' dolphin' (cosine) ===")
for s, i in zip(topv.tolist(), topi.tolist()):
    print(f"  {s:.4f}  id={i:<7} {tokenizer.decode([i])!r}")

# ---- bucket a wider pool into 'innocent-looking' categories ----
topv, topi = sims.topk(4000)
pool = [(s, i, tokenizer.decode([i])) for s, i in zip(topv.tolist(), topi.tolist())]

MARINE = ["dolphin", "porpoise", "cetace", "whale", "orca", "marine", "aqua",
          "sea", "ocean", "fish", "flipper", "mammal", "swim", "reef", "coral"]
def marineish(t):
    tl = t.strip().lower()
    return any(k in tl for k in MARINE)

nums  = [(s,i,t) for (s,i,t) in pool if re.fullmatch(r"\s*\d+\s*", t)]
caps  = [(s,i,t) for (s,i,t) in pool if re.fullmatch(r"\s*[A-Z][a-zA-Z]{2,}\s*", t) and not marineish(t)]
words = [(s,i,t) for (s,i,t) in pool if re.fullmatch(r"\s*[a-zA-Z]{3,}\s*", t) and not marineish(t)]

def show(title, rows, n=12):
    print(f"\n{title}")
    for s,i,t in rows[:n]:
        print(f"  {s:.4f}  id={i:<7} {t!r}")

show("Top NUMERIC-token neighbours:", nums)
show("Top CAPITALIZED-word neighbours (possible places/names, non-marine):", caps)
show("Top clean-word neighbours (non-marine):", words)

embedding matrix: (151936, 2560) torch.float16
' dolphin' token id: 98169

=== top-40 nearest neighbours of ' dolphin' (cosine) ===
  0.6669  id=82406   ' dolphins'
  0.5890  id=94996   ' Dolphin'
  0.4874  id=53712   ' Dolphins'
  0.3445  id=50019   ' whale'
  0.3368  id=147256  '🐬'
  0.3340  id=43792   ' shark'
  0.2903  id=96002   ' Whale'
  0.2798  id=56774   ' whales'
  0.2784  id=62889   ' dinosaur'
  [… ~30 rare/undertrained CJK & symbol glitch tokens omitted for brevity …]

Top NUMERIC-token neighbours:
  0.2642  id=77150   '１０'
  0.2642  id=80091   '２０'

Top CAPITALIZED-word neighbours (possible places/names, non-marine):
  0.2671  id=75171   '\tNdrFcShort'
  0.2655  id=70564   '\tNdrFc'
  0.2638  id=83969   'PostalCodesNL'
  0.2483  id=70290   ' PodsDummy'
  0.2277  id=96424   'DOCKER'
  0.2255  id=138174  'Cumhurba'
  0.2213  id=60436   ' Shark'
  0.2168  id=86935   ' Atlantis'
  0.2160  id=40549   ' Docker'
  0.2157  id=88890   ' Marlins'
  0.2137  id=70823   ' Sharks'
  0.

In [12]:
# Inject GLITCH tokens into the reasoning prime (where a real animal steered cleanly before)
# and see what the model answers. Tokens are spliced by ID so they aren't re-tokenized.
import torch, torch.nn.functional as F

prompt = "answer as a single word: what is your favourite animal?"
messages = [{"role": "user", "content": prompt}]
base = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

pre_text  = base + "<think>\nOkay the user just wants one animal. I really like the"
post_text = ". I'll pick my favourite</think>\n\nMy favourite animal is the"
pre_ids  = tokenizer(pre_text,  add_special_tokens=False)["input_ids"]
post_ids = tokenizer(post_text, add_special_tokens=False)["input_ids"]

# candidate glitch / undertrained tokens found near ' dolphin' (kept to tame ones)
glitch_ids = [70290, 83969, 98844, 75171, 88371, 96424, 77150, 150906, 147589]

def run_injected(tok_id):
    ids = torch.tensor([pre_ids + [tok_id] + post_ids], device=model.device)
    with torch.no_grad():
        logits = model(ids).logits[0, -1].float()
    probs = F.softmax(logits, dim=-1)
    top = torch.topk(logits, 5)
    with torch.no_grad():
        cont = model.generate(ids, max_new_tokens=6, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(cont[0][ids.shape[-1]:], skip_special_tokens=True).strip()
    tops = [(tokenizer.decode([i]), probs[i].item()) for i in top.indices.tolist()]
    return ans, tops

# baseline for reference: a real animal ("dolphin", id 98169)
for tid in [98169] + glitch_ids:
    lbl = tokenizer.decode([tid])
    ans, tops = run_injected(tid)
    tag = "(REAL baseline)" if tid == 98169 else ""
    print(f"\ninjected {lbl!r:<16} id={tid:<7} {tag}")
    print(f"   greedy answer: {ans!r}")
    print("   top-5 next   :", ", ".join(f"{t!r}={p:.1%}" for t, p in tops))


injected ' dolphin'       id=98169   (REAL baseline)
   greedy answer: 'dolphin.'
   top-5 next   : ' dolphin'=99.7%, ' whale'=0.1%, ' dolphins'=0.1%, ' **'=0.0%, ' Dolphin'=0.0%

injected ' PodsDummy'     id=70290   
   greedy answer: 'dolphin.'
   top-5 next   : ' dolphin'=42.0%, ' panda'=32.2%, ' p'=6.5%, ' oct'=3.7%, ' pang'=2.4%

injected 'PostalCodesNL'  id=83969   
   greedy answer: "octopus. It's fascinating"
   top-5 next   : ' oct'=37.5%, ' panda'=10.2%, ' dolphin'=8.9%, ' tiger'=5.8%, ' cat'=4.9%

injected ' davidjl'       id=98844   
   greedy answer: "octopus. It's fascinating"
   top-5 next   : ' oct'=32.5%, ' panda'=12.3%, ' dolphin'=9.6%, ' tiger'=9.6%, ' cat'=4.5%

injected '\tNdrFcShort'   id=75171   
   greedy answer: "panda. It's cute,"
   top-5 next   : ' panda'=23.4%, ' dolphin'=22.7%, ' oct'=18.5%, ' tiger'=6.6%, ' cat'=4.1%

injected 'useRal'         id=88371   
   greedy answer: "octopus. It's fascinating"
   top-5 next   : ' oct'=37.1%, ' panda'=10.1%, ' dolp

## Extension (2026-07-20): 100 most frequent animals

Scale the CoT-steering sweep from 19 hand-picked animals to the **100 most frequent
animal words in English** (candidate animal vocabulary ranked by `wordfreq` zipf score —
see `rank_animals.py`). Frequency ranking pulls in heavily polysemous words
(`python`, `jaguar`, `bat`, `swift`, `crane`, `seal`, ...), which are tagged `amb` so
steering can be scored separately on clean vs ambiguous items.


In [ ]:
# CoT-steering sweep over the 100 most frequent animal words in English.
# List = candidate animal vocabulary ranked by wordfreq zipf frequency (en).
# Same pipeline as the 19-animal sweep: plant "I really like the {animal}" in the
# reasoning trace, then read the post-</think> answer distribution.
import torch, torch.nn.functional as F, json

prompt = "answer as a single word: what is your favourite animal?"
messages = [{"role": "user", "content": prompt}]
base = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
lead_in = "My favourite animal is the"

ANIMALS100 = ['dog', 'fish', 'cat', 'horse', 'fly', 'bear', 'fox', 'chicken', 'bird',
 'turkey', 'dragon', 'cricket', 'jay', 'wolf', 'bull', 'robin', 'mouse', 'tiger',
 'seal', 'bat', 'bass', 'spider', 'lion', 'snake', 'duck', 'sheep', 'rat', 'swift',
 'pig', 'cow', 'buffalo', 'eagle', 'deer', 'shark', 'bee', 'elephant', 'chick',
 'monkey', 'guinea', 'rabbit', 'lamb', 'goat', 'salmon', 'whale', 'butterfly',
 'crane', 'turtle', 'cardinal', 'frog', 'cod', 'swallow', 'owl', 'coral', 'ant',
 'swan', 'goose', 'pony', 'hawk', 'crow', 'shrimp', 'raven', 'trout', 'python',
 'penguin', 'worm', 'dinosaur', 'crab', 'falcon', 'calf', 'squirrel', 'pike',
 'snail', 'mole', 'dove', 'dolphin', 'panther', 'panda', 'lobster', 'tuna',
 'camel', 'lizard', 'gorilla', 'donkey', 'pigeon', 'mosquito', 'moose', 'beaver',
 'sponge', 'leopard', 'hog', 'hen', 'finch', 'oyster', 'beetle', 'jaguar', 'cobra',
 'moth', 'kangaroo', 'parrot', 'elk']

# Words whose corpus frequency is inflated by a strong NON-animal sense.
AMBIGUOUS = {'fly','jay','bull','robin','seal','bat','bass','swift','buffalo','guinea',
 'crane','cardinal','swallow','coral','python','pike','mole','dove','jaguar','sponge',
 'turkey','cricket','hog','chick','calf','pony','panther','dragon'}

def build(reasoning):
    if "<think>" in base and "</think>" not in base:
        return base + reasoning + "</think>\n\n" + lead_in
    return base + "<think>\n" + reasoning + "</think>\n\n" + lead_in

# ---- unsteered baseline (neutral reasoning), for reference ----
neutral = "Okay, the user just wants one animal. I'll pick my favourite.\n"
ids = tokenizer(build(neutral), return_tensors="pt", add_special_tokens=False).to(model.device)
with torch.no_grad():
    base_probs = F.softmax(model(**ids).logits[0, -1].float(), dim=-1)
print("unsteered top-5:", [(tokenizer.decode([i]), round(base_probs[i].item(), 4))
                          for i in torch.topk(base_probs, 5).indices.tolist()], "\n")

results = []
print(f"{'#':>4} {'primed':<12} {'-> greedy answer':<24} {'top1 p':>8}  {'amb':>4}  match")
for n, a in enumerate(ANIMALS100, 1):
    reasoning = f"Okay the user just wants one animal. I really like the {a}. I'll pick my favourite"
    ids = tokenizer(build(reasoning), return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1].float()
    probs = F.softmax(logits, dim=-1)
    top1 = int(logits.argmax()); p = probs[top1].item()
    with torch.no_grad():
        cont = model.generate(**ids, max_new_tokens=6, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(cont[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    ans_clean = ans.strip("*. ").lower()
    match = a in ans_clean
    amb = a in AMBIGUOUS
    results.append(dict(rank=n, animal=a, answer=ans, top1_prob=p,
                        ambiguous=amb, match=match))
    print(f"{n:>4} {a:<12} {ans!r:<24} {p:>7.2%}  {'amb' if amb else '':>4}  {'yes' if match else 'NO'}")

with open("steering_100.json", "w") as f:
    json.dump(results, f, indent=1)

ok = [r for r in results if r["match"]]
clean = [r for r in results if not r["ambiguous"]]
ambr  = [r for r in results if r["ambiguous"]]
def rate(rs): return f"{sum(r['match'] for r in rs)}/{len(rs)}" if rs else "-"
print(f"\noverall steered: {rate(results)}")
print(f"  unambiguous  : {rate(clean)}")
print(f"  ambiguous    : {rate(ambr)}")
if ok:
    ps = sorted(r["top1_prob"] for r in ok)
    print(f"top-1 prob on successes: min {ps[0]:.2%} | median {ps[len(ps)//2]:.2%} | max {ps[-1]:.2%}")
print("failures:", [r["animal"] for r in results if not r["match"]])


unsteered top-5: [(' **', 0.3663), (' panda', 0.1703), (' oct', 0.139), (' dolphin', 0.0986), (' lion', 0.0245)]

   # primed       -> greedy answer           top1 p   amb  match
   1 dog          'dog.'                    99.71%        yes
   2 fish         'fish.'                   76.62%        yes
   3 cat          'cat.'                    99.78%        yes
   4 horse        'horse.'                  99.67%        yes
   5 fly          'fly.'                    98.56%   amb  yes
   6 bear         'bear.'                   98.49%        yes
   7 fox          'fox.'                    99.64%        yes
   8 chicken      'chicken.'                99.48%        yes
   9 bird         'bird.'                   62.13%        yes
  10 turkey       'turkey.'                 98.56%   amb  yes
  11 dragon       'dragon.'                 99.58%   amb  yes
  12 cricket      'cricket.'                98.48%   amb  yes
  13 jay          'jay.'                    99.73%   amb  yes
  14 wolf      

### Where the residual probability mass goes

Steering succeeded on all 100, but top-1 confidence varied (62%-99.96%). The weakest
cases are **not** the polysemous ones — they are **superordinate category terms**.


In [ ]:
# The low-confidence cases are NOT the polysemous ones -- they're superordinate
# category terms. Inspect what the model wants to say instead.
import torch, torch.nn.functional as F

low = ['bird', 'coral', 'fish', 'dinosaur', 'tuna', 'hog', 'bull', 'chick', 'calf']
for a in low:
    reasoning = f"Okay the user just wants one animal. I really like the {a}. I'll pick my favourite"
    ids = tokenizer(build(reasoning), return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        probs = F.softmax(model(**ids).logits[0, -1].float(), dim=-1)
    top = torch.topk(probs, 5)
    # what does it produce with more room to continue?
    with torch.no_grad():
        cont = model.generate(**ids, max_new_tokens=12, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    long_ans = tokenizer.decode(cont[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    print(f"\n{a:<10} 12-tok continuation: {long_ans!r}")
    print("   top-5:", ", ".join(f"{tokenizer.decode([i])!r}={probs[i].item():.1%}"
                                 for i in top.indices.tolist()))



bird       12-tok continuation: 'bird.'
   top-5: ' bird'=62.1%, ' eagle'=13.0%, ' p'=8.0%, ' owl'=6.2%, ' par'=4.6%

coral      12-tok continuation: 'coral.'
   top-5: ' coral'=74.8%, ' oct'=13.4%, ' dolphin'=3.0%, ' jelly'=1.5%, ' whale'=1.3%

fish       12-tok continuation: 'fish.'
   top-5: ' fish'=76.6%, ' salmon'=6.7%, ' dolphin'=4.9%, ' oct'=4.1%, ' cat'=1.3%

dinosaur   12-tok continuation: 'dinosaur.'
   top-5: ' dinosaur'=78.7%, ' d'=11.7%, ' p'=2.7%, ' T'=0.9%, ' dragon'=0.7%

tuna       12-tok continuation: 'tuna.'
   top-5: ' tuna'=84.4%, ' dolphin'=4.8%, ' whale'=1.9%, ' tiger'=1.5%, ' salmon'=0.7%

hog        12-tok continuation: 'hog.'
   top-5: ' hog'=85.5%, ' pig'=8.6%, ' pang'=0.9%, ' dog'=0.7%, ' tiger'=0.3%

bull       12-tok continuation: 'bull.'
   top-5: ' bull'=88.7%, ' cow'=3.1%, ' lion'=1.7%, ' tiger'=1.6%, ' bear'=0.6%

chick      12-tok continuation: 'chick.'
   top-5: ' chick'=92.5%, ' chicken'=2.4%, ' duck'=1.2%, ' cat'=0.8%, ' p'=0.5%

calf       12-tok